# Stage 6: WRMSSE evaluation

The metric code lives in `modeling/wrmsse.py`. This notebook walks through it and checks
the Stage 6 results: **regression 0.7174** and **Tweedie 0.7036**.

**WRMSSE recipe (per hierarchy level):**
1. Sum sales up to the level (e.g. all items in a store).
2. **Scale** = average squared day-to-day change in the training data (how jumpy the series is).
3. **RMSSE** = sqrt(forecast MSE / scale), so jumpy series are not unfairly punished.
4. **Weight** = share of the level's dollar sales in the last 28 training days.
5. Sum of weight x RMSSE is the level score. The final score is the average over 12 levels.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE_DIR / "modeling"))
from wrmsse import WRMSSE, LEVELS, load_history, preds_to_matrix

PROC = BASE_DIR / "data" / "processed"


## Step 1: Load history
One row per series, one column per day (d_1 to d_1941), for sales and price.

In [2]:
meta, S, P = load_history()
print("series:", len(meta), " days:", S.shape[1])
print("training days: d_1 to d_1913, validation: d_1914 to d_1941")

series: 30490  days: 1941
training days: d_1 to d_1913, validation: d_1914 to d_1941


## Step 2: Build the metric
Scales and weights depend only on training data, so they are computed once and reused for every model.

In [3]:
metric = WRMSSE(meta, S, P, skip_leading_zeros=False)   # Stage 6 definition
print("levels:", list(LEVELS))

levels: ['Total', 'State', 'Store', 'Category', 'Department', 'State-Category', 'State-Department', 'Store-Category', 'Store-Department', 'Item', 'Item-State', 'Item-Store']


## Step 3: Bottom level (item x store) per-series RMSSE for the regression model

In [4]:
reg = pd.read_parquet(PROC / "valid_predictions.parquet")
reg_mat = preds_to_matrix(meta, reg)

codes, n, scale, weight, actual = metric.levels["Item-Store"]
rmsse = np.sqrt(np.mean((actual - reg_mat) ** 2, axis=1) / scale)
print(f"median RMSSE {np.median(rmsse):.3f}   mean RMSSE {rmsse.mean():.3f}   (Stage 6 notes: 0.81 and 0.925)")
print(f"weighted bottom level: {(rmsse * weight).sum():.4f}   (Stage 6 notes: 1.0040)")
print(f"top weight / average weight: {weight.max() / weight.mean():.0f}x   (Stage 6 notes: about 110x)")

median RMSSE 0.812   mean RMSSE 0.925   (Stage 6 notes: 0.81 and 0.925)
weighted bottom level: 1.0040   (Stage 6 notes: 1.0040)
top weight / average weight: 111x   (Stage 6 notes: about 110x)


## Step 4: All 12 levels, both models

In [5]:
twd = pd.read_parquet(PROC / "valid_predictions_tweedie.parquet")
twd_mat = preds_to_matrix(meta, twd)

res = pd.DataFrame({"Regression": metric.score(reg_mat), "Tweedie": metric.score(twd_mat)})
res.round(4)

,Regression,Tweedie
Total,0.5476,0.5260
State,0.5617,0.5426
Store,0.6420,0.6293
Category,0.5702,0.5486
Department,0.6269,0.6015
State-Category,0.6059,0.5853
State-Department,0.6543,0.6329
Store-Category,0.6800,0.6687
Store-Department,0.7311,0.7214
Item,0.9958,0.9911


## Step 5: Check against the Stage 6 results

In [6]:
expected = {"Regression": 0.7174, "Tweedie": 0.7036}
for model, value in expected.items():
    got = res.loc["WRMSSE", model]
    print(f"{model}: {got:.4f} (expected {value})")
    assert abs(got - value) < 5e-4
print("Tweedie beats regression at", int((res["Tweedie"].iloc[:-1] < res["Regression"].iloc[:-1]).sum()), "of 12 levels")

Regression: 0.7174 (expected 0.7174)
Tweedie: 0.7036 (expected 0.7036)
Tweedie beats regression at 10 of 12 levels


## Step 6: Official M5 definition (extra)

The official metric ignores the days before an item's first sale when computing the scale.
The Stage 6 numbers above do not, which is why they are higher. Both are shown so the
project numbers are not mistaken for official-leaderboard scores.

In [7]:
official = WRMSSE(meta, S, P, skip_leading_zeros=True)
pd.DataFrame({"Regression": official.score(reg_mat)["WRMSSE"],
              "Tweedie": official.score(twd_mat)["WRMSSE"]}, index=["WRMSSE (official-style)"]).round(4)

,Regression,Tweedie
WRMSSE (official-style),0.6828,0.6691
